# Notebook 3 — Deployment for Physical COTS UEs

Loads a completed MGEN run, substitutes generated placeholder IPs with
real UE IPs, and produces ready-to-copy SSH/SCP commands for every node.

```
traffic_profiles/run_<apps>_<ts>/mgen_scripts/   (Notebook 2 output)
        ↓
  ★ THIS CODE ★
        ↓
traffic_profiles/run_<apps>_<ts>/deployment/
    ├── updated_scripts/     ← .mgn files with real UE IPs
    ├── DEPLOYMENT_GUIDE.md
    ├── dn_commands.txt
    └── nuc*_commands.txt
```

## Cell 1 — Configuration & UE Mapping

**Edit this cell before running anything else.**

### `UE_NAME_MAP` must be complete

### One generated UE per physical box — strictly enforced

Mapping two generated UEs to the same physical box is **not supported**.
All DL receivers must bind the same UDP port on the same interface, which
MGEN cannot do. Cell 1 raises an error if any physical box appears more
than once in `UE_NAME_MAP`.

In [2]:
import json
import shutil
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import pandas as pd

warnings.filterwarnings("ignore")

# ── helpers ───────────────────────────────────────────────────────────────────

def find_project_root() -> Path:
    current = Path.cwd()
    for cand in [current, *current.parents]:
        if (cand / "data").is_dir() and (cand / "artifacts").is_dir():
            return cand
    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT         = find_project_root()
TRAFFIC_PROFILES_DIR = PROJECT_ROOT / "traffic_profiles"

# ══════════════════════════════════════════════════════════════════════════════
#  SELECT RUN
#  OPTION 1 — auto: latest run that has an mgen_scripts/ folder
#  OPTION 2 — by index (uncomment)
#  OPTION 3 — by name  (uncomment)
# ══════════════════════════════════════════════════════════════════════════════

available_runs = sorted(
    d for d in TRAFFIC_PROFILES_DIR.iterdir()
    if d.is_dir()
    and d.name.startswith("run_")
    and (d / "mgen_scripts").exists()
)

if not available_runs:
    raise FileNotFoundError(
        "No MGEN runs found. Run Notebook 2 first."
    )

# OPTION 1 — latest
RUN_DIR = available_runs[-1]


# OPTION 2 — by index
# RUN_DIR = available_runs[0]


# OPTION 3 — by name
# RUN_DIR = TRAFFIC_PROFILES_DIR / "run_filimo_20260402_141942"

MGEN_SCRIPTS_DIR = RUN_DIR / "mgen_scripts"


# ══════════════════════════════════════════════════════════════════════════════
#  LOAD TESTBED CONFIG
#  All testbed-specific parameters live in testbed_config.yaml.
# ══════════════════════════════════════════════════════════════════════════════

import yaml

_tb_path = PROJECT_ROOT / "testbed_config.yaml"
if not _tb_path.exists():
    raise FileNotFoundError(
        f"testbed_config.yaml not found at {_tb_path}\n"
        "Copy it from the repo root next to this notebook."
    )

with open(_tb_path) as _f:
    _tb = yaml.safe_load(_f)

# ── Deployment flags ─────────────────────────────────────────────────────────

ALLOW_PLACEHOLDER_HOSTS = bool((_tb.get("flags") or {}).get("allow_placeholder_hosts", False))
ALLOW_INVALID_RUN       = bool((_tb.get("flags") or {}).get("allow_invalid_run",       False))

# ── DN config ─────────────────────────────────────────────────────────────────

_dn          = _tb["dn"]
CN5G_SSH_HOST = str(_dn["ssh_host"])
DN_CONFIG = {
    "container": str(_dn["container"]),
    "ssh_host" : CN5G_SSH_HOST,
    "ip"       : None,          # filled from run config.json below
    "mgen_dir" : str(_dn["mgen_dir"]),
}

# ── UE host settings ──────────────────────────────────────────────────────────

_ues          = _tb["ues"]
UE_USERNAME   = str(_ues["username"])
UE_MGEN_BASE_DIR = str(_ues["mgen_dir"])

PHYSICAL_UES: Dict[str, Dict] = {
    box_name: {
        "ssh_host" : str(box["ssh_host"]),
        "ip"       : str(box["ip"]),
        "interface": str(box["interface"]),
    }
    for box_name, box in _ues["boxes"].items()
}

# ── UE name map ───────────────────────────────────────────────────────────────

UE_NAME_MAP: Dict[str, str] = {
    str(gen): str(phy)
    for gen, phy in (_tb.get("ue_name_map") or {}).items()
}

print(f"  Testbed config : {_tb_path.relative_to(PROJECT_ROOT)}")
print(f"  Testbed label  : {_tb.get('testbed', 'unspecified')}")

# ── read run config ───────────────────────────────────────────────────────────

with open(RUN_DIR / "config.json") as f:
    _cfg = json.load(f)

DN_CONFIG["ip"] = _cfg["network"]["dn_ip"]
DL_PORT             = _cfg["network"]["dl_port"]
UL_PORT             = _cfg["network"]["ul_port"]
SIMULATION_DURATION = _cfg["simulation_duration"]
APPS                = _cfg["apps"]

# ── Notebook 1 validation gate ────────────────────────────────────────────────

_nb1_validation = _cfg.get("validation", {})
_nb1_passed     = _nb1_validation.get("passed", True)
if not _nb1_passed and not ALLOW_INVALID_RUN:
    _nb1_errors = _nb1_validation.get("errors", [])
    _preview    = "\n".join(f"  • {e}" for e in _nb1_errors[:5])
    _extra      = f"  ... and {len(_nb1_errors)-5} more" if len(_nb1_errors) > 5 else ""
    raise ValueError(
        f"Run '{RUN_DIR.name}' failed Notebook 1 validation "
        f"({len(_nb1_errors)} error(s)):\n{_preview}\n{_extra}\n"
        "Fix the run in Notebook 1, or set allow_invalid_run: true "
        "in testbed_config.yaml to proceed anyway."
    )
elif _nb1_passed:
    print("  ✅  Notebook 1 validation passed")
else:
    print("  ⚠️   Notebook 1 validation FAILED — proceeding anyway "
          "(allow_invalid_run: true in testbed_config.yaml)")

# ── load manifest ─────────────────────────────────────────────────────────────

manifest_df     = pd.read_csv(MGEN_SCRIPTS_DIR / "manifest.csv")
generated_names = set(manifest_df["ue_name"].tolist())

# ── validate UE_NAME_MAP ──────────────────────────────────────────────────────

unmapped = generated_names - set(UE_NAME_MAP.keys())
if unmapped:
    raise ValueError(
        f"ue_name_map in testbed_config.yaml is incomplete.\n"
        f"  Missing generated UEs: {sorted(unmapped)}\n"
        f"  All generated UEs    : {sorted(generated_names)}\n"
        "Add each missing UE to ue_name_map in testbed_config.yaml."
    )

phantom = set(UE_NAME_MAP.keys()) - generated_names
if phantom:
    raise ValueError(
        f"ue_name_map references generated UEs not in the manifest: "
        f"{sorted(phantom)}\n"
        f"Available: {sorted(generated_names)}"
    )

# Physical boxes referenced must exist — check BEFORE resolving SSH hosts

unknown_physical = set(UE_NAME_MAP.values()) - set(PHYSICAL_UES.keys())
if unknown_physical:
    raise ValueError(
        f"ue_name_map references physical boxes not in testbed_config.yaml: "
        f"{sorted(unknown_physical)}"
    )

# One generated UE per physical box only

from collections import Counter as _Counter
_box_counts = _Counter(UE_NAME_MAP.values())
_multi      = {box: n for box, n in _box_counts.items() if n > 1}
if _multi:
    raise ValueError(
        "Multiple generated UEs mapped to one physical box is not supported.\n"
        "All DL receivers must bind the same UDP port on the same host.\n"
        f"Conflicts: {_multi}\n"
        "Fix ue_name_map in testbed_config.yaml."
    )

# Validate SSH hostnames — only used boxes

used_boxes      = set(UE_NAME_MAP.values())
_hosts_to_check = [CN5G_SSH_HOST] + [
    PHYSICAL_UES[name]["ssh_host"] for name in used_boxes
]
_bad_hosts = [h for h in _hosts_to_check if "<experiment>" in h]
if _bad_hosts and not ALLOW_PLACEHOLDER_HOSTS:
    raise ValueError(
        "Replace <experiment> in testbed_config.yaml SSH hostnames:\n"
        + "\n".join(f"  - {h}" for h in _bad_hosts)
        + "\n\nTip: ssh <host> hostname to verify the exact FQDN.\n"
        + "Or set allow_placeholder_hosts: true in testbed_config.yaml "
        + "while pre-staging."
    )
elif _bad_hosts:
    print(f"  ⚠️   {len(_bad_hosts)} SSH hostname(s) still contain <experiment> "
          f"(allow_placeholder_hosts: true — fix before deploying)")

# Duplicate physical IP check

_used_ips = [PHYSICAL_UES[name]["ip"] for name in used_boxes]
_dup_ips  = sorted({ip for ip in _used_ips if _used_ips.count(ip) > 1})
if _dup_ips:
    raise ValueError(
        f"Duplicate physical UE IP(s) in testbed_config.yaml: {_dup_ips}\n"
        "Each active box must have a unique IP."
    )

# ── build per-generated-UE mapping ───────────────────────────────────────────

ue_mapping: List[Dict] = []
for gen_name, phy_name in UE_NAME_MAP.items():
    gen_row = manifest_df[manifest_df["ue_name"] == gen_name].iloc[0]
    phy     = PHYSICAL_UES[phy_name]
    ue_mapping.append({
        "generated_ue_name"    : gen_name,
        "generated_ue_ip"      : gen_row["ue_ip"],
        "physical_ue_name"     : phy_name,
        "physical_ue_ssh_host" : phy["ssh_host"],
        "physical_ue_ip"       : phy["ip"],
        "physical_ue_interface": phy["interface"],
        "dl_rx_script"         : gen_row["dl_rx_script"],
        "ul_tx_script"         : gen_row["ul_tx_script"],
        "ue_class"             : gen_row["ue_class"],
        "n_dl_events"          : gen_row["n_dl_events"],
        "n_ul_events"          : gen_row["n_ul_events"],
    })

box_to_ues: Dict[str, List[Dict]] = defaultdict(list)
for m in ue_mapping:
    box_to_ues[m["physical_ue_name"]].append(m)

# ── print summary ─────────────────────────────────────────────────────────────

print("="*72)
print("  DEPLOYMENT CONFIGURATION")
print("="*72)
print(f"\n  Run      : {RUN_DIR.name}")
print(f"  Apps     : {', '.join(APPS)}")
print(f"  Duration : {SIMULATION_DURATION}s  "
      f"({SIMULATION_DURATION/60:.1f} min)")
print(f"  Ports    : DL={DL_PORT}  UL={UL_PORT}")
print(f"\n  CN5G host    : {CN5G_SSH_HOST}")
print(f"  Container    : {DN_CONFIG['container']}  "
      f"(DN IP: {DN_CONFIG['ip']})")
print(f"  UE base dir  : {UE_MGEN_BASE_DIR}  (absolute)")

print(f"\n  UE Mapping — {len(ue_mapping)} generated UEs, "
      f"{len(box_to_ues)} physical boxes:")
for m in ue_mapping:
    print(f"\n    {m['generated_ue_name']} ({m['ue_class']}) "
          f"→ {m['physical_ue_name'].upper()}")
    print(f"       IP        : "
          f"{m['generated_ue_ip']} → {m['physical_ue_ip']}")
    print(f"       interface : {m['physical_ue_interface']}")
    print(f"       SSH       : {m['physical_ue_ssh_host']}")
    print(f"       scripts   : "
          f"{m['dl_rx_script']}  {m['ul_tx_script']}")
    print(f"       events    : "
          f"{m['n_dl_events']:,} DL  {m['n_ul_events']:,} UL")

print("\n  ✅  Cell 1 complete — all validation passed")
print("  Run Cell 2 to generate deployment files")

  ℹ️  Multiple generated UEs on one physical box:
     nuc4 ← ['ue4', 'ue5', 'ue6']
  Each pair of scripts needs its own terminal on that box.
  DEPLOYMENT CONFIGURATION

  Run      : run_filimo_igap_aparat_telegram_youtube_20260402_141942
  Apps     : filimo, igap, aparat, telegram, youtube
  Duration : 600s  (10.0 min)
  Ports    : DL=5000  UL=6000

  CN5G host    : ghinwa@cn5g-docker-host.<experiment>.emulab.net
  Container    : oai-ext-dn  (DN IP: 192.168.70.163)
  UE base dir  : /home/ghinwa/mgen_runs  (absolute)

  UE Mapping — 6 generated UEs, 4 physical boxes:

    ue1 (heavy) → NUC1
       IP        : 12.1.1.1 → 12.1.1.136
       interface : qwwan0
       SSH       : ghinwa@ota-nuc1-cots-ue.<experiment>.emulab.net
       scripts   : ue1_dl_rx.mgn  ue1_ul_tx.mgn
       events    : 2,420 DL  1,692 UL

    ue2 (heavy) → NUC2
       IP        : 12.1.1.2 → 12.1.1.139
       interface : qwwan0
       SSH       : ghinwa@ota-nuc2-cots-ue.<experiment>.emulab.net
       scripts   : ue2_

## Cell 2 — Generate Deployment Files

1. Pre-flight: verifies all expected scripts exist
2. Rewrites `dn_dl_tx.mgn` with real UE IPs for **all** generated UEs
3. Copies UE scripts unchanged
4. Writes per-physical-box `.txt` manual command guides (1:1 UE mapping)
5. Writes `DEPLOYMENT_GUIDE.md`

In [3]:
# ── output directories ────────────────────────────────────────────────────────

deployment_dir      = RUN_DIR / "deployment"
updated_scripts_dir = deployment_dir / "updated_scripts"
deployment_dir.mkdir(exist_ok=True)
updated_scripts_dir.mkdir(exist_ok=True)

ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name    = f"mgen_{ts}"
scripts_abs = updated_scripts_dir.resolve()
dur_s       = SIMULATION_DURATION
dur_min     = dur_s / 60

print("="*72)
print("  GENERATING DEPLOYMENT FILES")
print("="*72)
print(f"\n  Output dir     : "
      f"{deployment_dir.relative_to(PROJECT_ROOT)}")
print(f"  Remote run name: {run_name}")

# ══════════════════════════════════════════════════════════════════════════════
# verify all required scripts exist before writing anything
# ══════════════════════════════════════════════════════════════════════════════

print("\n  Pre-flight checks...")

required = (
    [MGEN_SCRIPTS_DIR / "dn_dl_tx.mgn",
     MGEN_SCRIPTS_DIR / "dn_ul_rx.mgn"]
    + [MGEN_SCRIPTS_DIR / m["dl_rx_script"] for m in ue_mapping]
    + [MGEN_SCRIPTS_DIR / m["ul_tx_script"] for m in ue_mapping]
)

missing = [p for p in required if not p.exists()]
if missing:
    for p in missing:
        print(f"  ❌  Missing: {p.relative_to(PROJECT_ROOT)}")
    raise FileNotFoundError(
        f"{len(missing)} script(s) missing — re-run Notebook 2."
    )

print(f"  ✅  All {len(required)} required scripts present")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — Rewrite dn_dl_tx.mgn with real UE IPs
#
#  Every generated UE is in UE_NAME_MAP (validated in Cell 1), so every
#  placeholder IP gets replaced. No placeholder IPs remain after this step.
# ══════════════════════════════════════════════════════════════════════════════

print("\n  [1/4] Rewriting dn_dl_tx.mgn with real UE IPs...")

with open(MGEN_SCRIPTS_DIR / "dn_dl_tx.mgn") as fh:
    dn_dl_content = fh.read()

for m in ue_mapping:
    old = m["generated_ue_ip"]
    new = m["physical_ue_ip"]
    dn_dl_content = dn_dl_content.replace(f"DST {old}/", f"DST {new}/")
    print(f"    {old} → {new}")

# Verify no placeholder IPs remain

all_generated_ips = {m["generated_ue_ip"] for m in ue_mapping}
residual = [ip for ip in all_generated_ips if f"DST {ip}/" in dn_dl_content]
if residual:
    raise RuntimeError(
        f"Placeholder IPs still in dn_dl_tx.mgn after rewrite: {residual}\n"
        f"This should not happen — check UE_NAME_MAP."
    )

(updated_scripts_dir / "dn_dl_tx.mgn").write_text(dn_dl_content)
shutil.copy(MGEN_SCRIPTS_DIR / "dn_ul_rx.mgn",
            updated_scripts_dir / "dn_ul_rx.mgn")

for m in ue_mapping:
    for script in [m["dl_rx_script"], m["ul_tx_script"]]:
        shutil.copy(MGEN_SCRIPTS_DIR / script,
                    updated_scripts_dir / script)

n_scripts = len(list(updated_scripts_dir.glob("*.mgn")))
print(f"  ✅  {n_scripts} scripts in updated_scripts/  "
      f"(no placeholder IPs remain)")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — Build command strings
#
#  Commands are built per generated UE, then grouped by physical box.
#  When multiple generated UEs share a box, the command file lists all
#  their scripts — the operator runs each in its own terminal.
#
#  docker exec: no -it flag — all calls are non-interactive scripted
#  commands; -it causes tty errors through scripted SSH.
#
#  Paths: absolute throughout — tilde expansion is unreliable in
#  quoted ssh/scp commands across environments.
# ══════════════════════════════════════════════════════════════════════════════

print("\n  [2/4] Building deployment commands...")

cn5g   = DN_CONFIG["ssh_host"]
cname  = DN_CONFIG["container"]
dn_run = f"{DN_CONFIG['mgen_dir']}/{run_name}"

# ── DN command sections (one per section, not per UE) ─────────────────────────

ue_ips_str = ", ".join(m["physical_ue_ip"] for m in ue_mapping)

dn_cmds: Dict[str, str] = {}

dn_cmds["setup"] = f"""\
# ── DN SETUP        ─────────────────────────────────────────────
# Docker is on the remote CN5G host.
# Sequence: scp to CN5G host → docker cp into container.

# 1. Create directory inside the container (no -it: non-interactive)
ssh {cn5g} "sudo docker exec {cname} mkdir -p {dn_run}"

# 2. SCP scripts to CN5G host
scp "{scripts_abs}/dn_dl_tx.mgn" {cn5g}:/tmp/dn_dl_tx.mgn
scp "{scripts_abs}/dn_ul_rx.mgn" {cn5g}:/tmp/dn_ul_rx.mgn

# 3. Copy from CN5G host into container
ssh {cn5g} "sudo docker cp /tmp/dn_dl_tx.mgn {cname}:{dn_run}/"
ssh {cn5g} "sudo docker cp /tmp/dn_ul_rx.mgn {cname}:{dn_run}/"

# 4. Verify (no -it: non-interactive)
ssh {cn5g} "sudo docker exec {cname} ls -lh {dn_run}"
"""

dn_cmds["start_receiver"] = f"""\
# ── DN UPLINK RECEIVER  (start FIRST — before any UE sender) ──────────────────
# No interface flag needed — DN receives on all interfaces.
# Sends to: {ue_ips_str}
ssh {cn5g}
sudo docker exec {cname} bash -c \\
  'mgen input {dn_run}/dn_ul_rx.mgn output {dn_run}/dn_ul_rx.log'
# Leave this running.
"""

dn_cmds["start_sender"] = f"""\
# ── DN DOWNLINK SENDER  (start AFTER all UE receivers are ready) ──────────────
# No interface flag needed — DN has one relevant interface.
ssh {cn5g}
sudo docker exec {cname} bash -c \\
  'mgen txlog input {dn_run}/dn_dl_tx.mgn output {dn_run}/dn_dl_tx.log'
# Runs for {dur_s}s ({dur_min:.1f} min) then stops automatically.
"""

dn_cmds["check_logs"] = f"""\
# ── DN LOG CHECK ──────────────────────────────────────────────────────────────
ssh {cn5g}

# UL received (packets FROM UEs)
sudo docker exec {cname} bash -c \\
  'grep -c "RECV" {dn_run}/dn_ul_rx.log || echo 0'

# DL sent (packets TO UEs)
sudo docker exec {cname} bash -c \\
  'grep -c "SEND" {dn_run}/dn_dl_tx.log || echo 0'
"""

# ── per-physical-box command sections ────────────────────────────────────────
# Keyed by physical box name; each value is a dict of section → text.
# When a box hosts multiple generated UEs, all their scripts are listed.

box_cmds: Dict[str, Dict[str, str]] = {}

for phy_name, ues_on_box in box_to_ues.items():
    ssh   = PHYSICAL_UES[phy_name]["ssh_host"]
    iface = PHYSICAL_UES[phy_name]["interface"]
    ue_run = f"{UE_MGEN_BASE_DIR}/{run_name}"

    # ── setup: copy all scripts for this box ──────────────────────────────
    
    gen_labels = ", ".join(
        f"{u['generated_ue_name']} ({u['ue_class']})" for u in ues_on_box
    )
    setup_lines = [
        f"# ── {phy_name.upper()} SETUP  "
        f"(generated: {gen_labels})",
        f"# Run locally.",
        f"",
        f"# 1. Create run directory (absolute path)",
        f'ssh "{ssh}" "mkdir -p {ue_run}"',
        f"",
        f"# 2. Copy scripts",
    ]
    for u in ues_on_box:
        setup_lines += [
            f'scp "{scripts_abs}/{u["dl_rx_script"]}" "{ssh}:{ue_run}/"',
            f'scp "{scripts_abs}/{u["ul_tx_script"]}" "{ssh}:{ue_run}/"',
        ]
    setup_lines += [
        f"",
        f"# 3. Verify",
        f'ssh "{ssh}" "ls -lh {ue_run}"',
    ]

    # ── start receivers: one command per generated UE ─────────────────────
    
    rx_lines = [
        f"# ── {phy_name.upper()} DOWNLINK RECEIVERS  "
        f"(start FIRST — before DN sender)",
        f"# ⚠️  interface {iface} is REQUIRED on every command.",
        f"# Without it, MGEN binds to eth0 instead of {iface} "
        f"and receives 0 packets.",
    ]
    if len(ues_on_box) > 1:
        rx_lines.append(
            f"# Each receiver needs its own terminal on this box."
        )
    for u in ues_on_box:
        dl_log = u["dl_rx_script"].replace(".mgn", ".log")
        rx_lines += [
            f"",
            f"# {u['generated_ue_name']} ({u['ue_class']})",
            f'ssh "{ssh}"',
            f"sudo mgen interface {iface} \\\\",
            f"  input  {ue_run}/{u['dl_rx_script']} \\\\",
            f"  output {ue_run}/{dl_log}",
        ]

    # ── start senders: one command per generated UE ───────────────────────
    
    tx_lines = [
        f"# ── {phy_name.upper()} UPLINK SENDERS  "
        f"(start AFTER DN receiver is ready)",
        f"# ⚠️  interface {iface} is REQUIRED on every command.",
        f"# Without it, traffic routes via eth0 and the DN receives "
        f"0 packets.",
    ]
    if len(ues_on_box) > 1:
        tx_lines.append(
            f"# Each sender needs its own terminal on this box."
        )
    for u in ues_on_box:
        ul_log = u["ul_tx_script"].replace(".mgn", ".log")
        tx_lines += [
            f"",
            f"# {u['generated_ue_name']} ({u['ue_class']})",
            f'ssh "{ssh}"',
            f"sudo mgen txlog interface {iface} \\\\",
            f"  input  {ue_run}/{u['ul_tx_script']} \\\\",
            f"  output {ue_run}/{ul_log}",
            f"# Runs for {dur_s}s ({dur_min:.1f} min) then stops.",
        ]

    # ── check logs: one block per generated UE ────────────────────────────
    
    log_lines = [f"# ── {phy_name.upper()} LOG CHECK"]
    for u in ues_on_box:
        dl_log = u["dl_rx_script"].replace(".mgn", ".log")
        ul_log = u["ul_tx_script"].replace(".mgn", ".log")
        log_lines += [
            f"",
            f'ssh "{ssh}"',
            f"# {u['generated_ue_name']} — DL received (FROM DN)",
            f"grep -c 'RECV' {ue_run}/{dl_log} || echo 0",
            f"# {u['generated_ue_name']} — UL sent (TO DN)",
            f"grep -c 'SEND' {ue_run}/{ul_log} || echo 0",
        ]

    box_cmds[phy_name] = {
        "setup"          : "\n".join(setup_lines),
        "start_receivers": "\n".join(rx_lines),
        "start_senders"  : "\n".join(tx_lines),
        "check_logs"     : "\n".join(log_lines),
    }

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — Write command .txt manual guides
# ══════════════════════════════════════════════════════════════════════════════

print("\n  [3/4] Writing command files...")

def write_cmd_file(path: Path, title: str, sections: Dict[str, str]):
    with open(path, "w") as fh:
        fh.write(f"# {title}\n")
        fh.write(f"# Generated: {ts}\n")
        fh.write("#\n")
        fh.write("# ⚠️  MANUAL COMMAND GUIDE — NOT AN EXECUTABLE SCRIPT\n")
        fh.write("#\n")
        fh.write("# These commands are meant to be copied and run ONE BY ONE in separate\n")
        fh.write("# terminals. Running this file directly with `bash` will NOT work:\n")
        fh.write("# the `ssh host` lines open a session that closes immediately,\n")
        fh.write("# and the following commands execute locally rather than remotely.\n")
        fh.write("#\n")
        fh.write("# Follow the steps in DEPLOYMENT_GUIDE.md for the correct sequence.\n")
        fh.write("\n")
        for text in sections.values():
            fh.write(text + "\n\n")

write_cmd_file(
    deployment_dir / "dn_commands.txt",
    "DN Deployment Commands",
    dn_cmds,
)
print("    ✅  dn_commands.txt")

for phy_name, cmds in box_cmds.items():
    write_cmd_file(
        deployment_dir / f"{phy_name}_commands.txt",
        f"{phy_name.upper()} Deployment Commands",
        cmds,
    )
    print(f"    ✅  {phy_name}_commands.txt")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — Write DEPLOYMENT_GUIDE.md
# ══════════════════════════════════════════════════════════════════════════════

print("\n  [4/4] Writing DEPLOYMENT_GUIDE.md...")

guide_path = deployment_dir / "DEPLOYMENT_GUIDE.md"

with open(guide_path, "w") as fh:

    fh.write(f"""# MGEN Multi-UE Bidirectional Traffic — Deployment Guide

**Generated :** {ts}
**Run       :** {RUN_DIR.name}
**Apps      :** {', '.join(APPS)}
**Duration  :** {dur_s}s ({dur_min:.1f} min)
**UEs       :** {len(ue_mapping)} generated → {len(box_to_ues)} physical boxes

---

## Overview

Bidirectional traffic:
- **Downlink** DN → UEs on UDP port {DL_PORT}
- **Uplink**   UEs → DN on UDP port {UL_PORT}

### UE profile assignments

| Physical box | Generated UE | Class | DL events | UL events |
|---|---|---|---|---|
""")
    for m in ue_mapping:
        fh.write(
            f"| {m['physical_ue_name'].upper()} "
            f"| {m['generated_ue_name']} "
            f"| {m['ue_class']} "
            f"| {m['n_dl_events']:,} "
            f"| {m['n_ul_events']:,} |\n"
        )

    fh.write(f"""
---

### Critical: UE interface requirement

Every UE MGEN command **must** include `interface <ue_interface>`
(configured per box in `testbed_config.yaml`).
Without it, traffic uses the management Ethernet interface (eth0)
and never reaches the 5G core.

| Direction | Node | Interface flag |
|---|---|---|
| DL receiver | UE | ✅ `interface <ue_interface>` required |
| UL sender   | UE | ✅ `interface <ue_interface>` required |
| DL sender   | DN | ❌ not needed |
| UL receiver | DN | ❌ not needed |

### Critical: remote Docker copy

Docker runs on the CN5G host, not locally.
Always: `scp` to CN5G host → `docker cp` into container remotely.

### Note: `docker exec` flags

All `docker exec` calls omit `-it`. Those flags are for interactive
terminals and cause tty errors when used through scripted SSH.

---

## Step 1 — Copy scripts to all machines

Run all commands **locally**.

### DN

```bash
{dn_cmds['setup']}
```

### UEs
""")

    for phy_name, cmds in box_cmds.items():
        fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
        fh.write(cmds["setup"])
        fh.write("\n```\n")

    fh.write(f"""
---

## Step 2 — Start all receivers (FIRST)

Open a **separate terminal** for each receiver.
All receivers must be running before any sender starts.

### DN uplink receiver

```bash
{dn_cmds['start_receiver']}
```

### UE downlink receivers
""")

    for phy_name, cmds in box_cmds.items():
        fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
        fh.write(cmds["start_receivers"])
        fh.write("\n```\n")

    fh.write(f"""
---

## Step 3 — Wait 5–10 seconds

Let all receivers stabilise before starting senders.

---

## Step 4 — Start all senders (SECOND)

Open a **separate terminal** for each sender.

### DN downlink sender

```bash
{dn_cmds['start_sender']}
```

### UE uplink senders
""")

    for phy_name, cmds in box_cmds.items():
        fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
        fh.write(cmds["start_senders"])
        fh.write("\n```\n")

    fh.write(f"""
---

## Step 5 — Monitor

Traffic runs for **{dur_s}s ({dur_min:.1f} min)**.
Senders stop automatically. Stop receivers with `Ctrl+C` after.

---

## Step 6 — Check results

### DN

```bash
{dn_cmds['check_logs']}
```

### UEs
""")

    for phy_name, cmds in box_cmds.items():
        fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
        fh.write(cmds["check_logs"])
        fh.write("\n```\n")

    fh.write("""
---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| UE RECV = 0 (DL) | Missing `interface <ue_interface>` on receiver | Add `interface <ue_interface>` (from testbed_config.yaml) |
| DN RECV = 0 (UL) | UE sent via management interface | Add `interface <ue_interface>` to UE sender |
| SSH refused | Wrong FQDN (missing `-cots-ue`) | Run `ssh <host> hostname` to verify |
| `docker cp` fails from ... | Docker daemon is remote | SCP to CN5G host first, then `docker cp` |
| tty error on `docker exec` | `-it` in scripted context | Already fixed — no `-it` in these scripts |
| Traffic stops early | Last burst < SIMULATION_DURATION | Increase duration in Notebook 1 Cell 2 |
| Wrong class on wrong box | `UE_NAME_MAP` stale | Update `UE_NAME_MAP` in Notebook 3 Cell 1 |
| DN sends to placeholder IP | `UE_NAME_MAP` incomplete | Cell 1 now raises an error — fix the map |

---

## Files reference

- `DEPLOYMENT_GUIDE.md` — this file
- `dn_commands.txt` — all DN commands
""")
    for phy_name in box_cmds:
        fh.write(
            f"- `{phy_name}_commands.txt` — {phy_name.upper()} commands\n"
        )
    fh.write("- `updated_scripts/` — .mgn files with real UE IPs\n")

print("    ✅  DEPLOYMENT_GUIDE.md")

# ── final summary ─────────────────────────────────────────────────────────────

print(f"""
{'='*72}
  DEPLOYMENT PACKAGE READY
{'='*72}

  Location : {deployment_dir.relative_to(PROJECT_ROOT)}
  Start    : open DEPLOYMENT_GUIDE.md

  Contents:
""")
for item in sorted(deployment_dir.iterdir()):
    if item.is_file():
        print(f"    {item.name}")
    elif item.is_dir():
        count = len(list(item.glob("*.mgn")))
        print(f"    {item.name}/   ({count} .mgn scripts)")

print()
print("  ✅  Cell 2 complete")

  GENERATING DEPLOYMENT FILES

  Output dir     : traffic_profiles/run_filimo_igap_aparat_telegram_youtube_20260402_141942/deployment
  Remote run name: mgen_20260409_111620

  Pre-flight checks...
  ✅  All 14 required scripts present

  [1/4] Rewriting dn_dl_tx.mgn with real UE IPs...
    12.1.1.1 → 12.1.1.136
    12.1.1.2 → 12.1.1.139
    12.1.1.3 → 12.1.1.137
    12.1.1.4 → 12.1.1.138
    12.1.1.5 → 12.1.1.138
    12.1.1.6 → 12.1.1.138
  ✅  14 scripts in updated_scripts/  (no placeholder IPs remain)

  [2/4] Building deployment commands...

  [3/4] Writing command files...
    ✅  dn_commands.sh
    ✅  nuc1_commands.sh
    ✅  nuc2_commands.sh
    ✅  nuc3_commands.sh
    ✅  nuc4_commands.sh

  [4/4] Writing DEPLOYMENT_GUIDE.md...
    ✅  DEPLOYMENT_GUIDE.md

  DEPLOYMENT PACKAGE READY

  Location : traffic_profiles/run_filimo_igap_aparat_telegram_youtube_20260402_141942/deployment
  Start    : open DEPLOYMENT_GUIDE.md

  Contents:

    .ipynb_checkpoints/   (0 .mgn scripts)
    DEPLOYM